Explainable Transformer Based Framework for Early Detection of Mental Health Disorders from Social Media Text

---

### imports and paths

In [ ]:
import re, pickle
import pandas as pd
import nltk
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

RAW_DIR  = Path('..') / 'datasets' / 'raw'
PROC_DIR = Path('..') / 'datasets' / 'processed'
SPLITS   = PROC_DIR / 'splits'
SPLITS.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
print('paths configured')

---

### load dataset 1 — sentiment analysis for mental health

In [2]:
df1 = pd.read_csv(RAW_DIR / 'Combined Data.csv')
df1['statement'] = df1['statement'].fillna('').astype(str)
df1['status']    = df1['status'].fillna('').astype(str)

# standardise column names
df1 = df1[['statement', 'status']].rename(columns={'statement': 'text', 'status': 'label'})
df1['source'] = 'combined'

print(f'dataset 1: {len(df1):,} rows')
print('classes:', sorted(df1['label'].unique()))

dataset 1: 53,043 rows
classes: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']


---

### load dataset 2 — depression reddit cleaned

In [3]:
df2 = pd.read_csv(RAW_DIR / 'depression_dataset_reddit_cleaned.csv')
df2['clean_text']    = df2['clean_text'].fillna('').astype(str)
df2['is_depression'] = df2['is_depression'].astype(int)

# map binary labels to mental health classes
df2['label'] = df2['is_depression'].map({1: 'Depression', 0: 'Normal'})
df2 = df2[['clean_text', 'label']].rename(columns={'clean_text': 'text'})
df2['source'] = 'reddit'

print(f'dataset 2: {len(df2):,} rows')
print('classes:', sorted(df2['label'].unique()))

dataset 2: 7,731 rows
classes: ['Depression', 'Normal']


---

### merge and initial cleaning

In [4]:
df = pd.concat([df1, df2], ignore_index=True)
print(f'merged: {len(df):,} rows')

# drop Personality disorder
df = df[df['label'] != 'Personality disorder']

# merge Anxiety + Stress → Anxiety/Stress
df['label'] = df['label'].replace({'Anxiety': 'Anxiety/Stress', 'Stress': 'Anxiety/Stress'})

# drop empty text
df = df[df['text'].str.strip().str.len() > 0]

# drop duplicates on text
before = len(df)
df = df.drop_duplicates(subset='text')
print(f'after dropping duplicates: {len(df):,} rows (removed {before - len(df):,})')

# drop rows with very short text (< 5 words)
df['word_count'] = df['text'].str.split().str.len()
df = df[df['word_count'] >= 5].drop(columns='word_count')
df = df.reset_index(drop=True)
print(f'after removing short texts: {len(df):,} rows')

print('\nclass distribution after merge:')
print(df['label'].value_counts().to_string())

merged: 60,774 rows
after dropping duplicates: 50,178 rows (removed 9,157)


after removing short texts: 47,392 rows

class distribution after merge:
label
Depression        15013
Normal            13450
Suicidal          10606
Anxiety/Stress     5823
Bipolar            2500


---

### text preprocessing pipeline

In [5]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)     # remove URLs
    text = re.sub(r'[^\w\s]', '', text)               # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and t.isalpha()]
    return ' '.join(tokens)

print('preprocessing text — this may take a minute ...')
df['clean_text'] = df['text'].apply(preprocess)

# drop rows where preprocessing yields empty string
df = df[df['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'rows after preprocessing: {len(df):,}')

# preview
df[['text', 'clean_text', 'label']].head(3)

preprocessing text — this may take a minute ...


rows after preprocessing: 47,371


,text,clean_text,label
0,"trouble sleeping, confused mind, restless hear...",trouble sleeping confused mind restless heart ...,Anxiety/Stress
1,"All wrong, back off dear, forward doubt. Stay ...",wrong back dear forward doubt stay restless re...,Anxiety/Stress
2,I've shifted my focus to something else but I'...,ive shifted focus something else im still worried,Anxiety/Stress


---

### label encoding

In [6]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print('label mapping:')
for idx, cls in enumerate(le.classes_):
    print(f'  {idx} → {cls}')

# save encoder
with open(SPLITS / 'label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print('\nlabel_encoder.pkl saved')

label mapping:
  0 → Anxiety/Stress
  1 → Bipolar
  2 → Depression
  3 → Normal
  4 → Suicidal

label_encoder.pkl saved


---

### train / validation / test split

In [7]:
# 70 % train  |  15 % val  |  15 % test  — stratified
train, temp = train_test_split(
    df, test_size=0.30, random_state=RANDOM_SEED, stratify=df['label_encoded']
)
val, test = train_test_split(
    temp, test_size=0.50, random_state=RANDOM_SEED, stratify=temp['label_encoded']
)

print(f'train : {len(train):,}')
print(f'val   : {len(val):,}')
print(f'test  : {len(test):,}')

# save splits
train.to_csv(SPLITS / 'train.csv', index=False)
val.to_csv(SPLITS   / 'val.csv',   index=False)
test.to_csv(SPLITS  / 'test.csv',  index=False)

# save full cleaned dataset
df.to_csv(PROC_DIR / 'cleaned_data.csv', index=False)

print('\nsaved: train.csv, val.csv, test.csv, cleaned_data.csv')

train : 33,159
val   : 7,106
test  : 7,106



saved: train.csv, val.csv, test.csv, cleaned_data.csv


---

### summary

In [8]:
import os

print('processed files:')
for f in sorted(SPLITS.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<25} {size_kb:>8.1f} KB')

cleaned = PROC_DIR / 'cleaned_data.csv'
print(f'  {cleaned.name:<25} {cleaned.stat().st_size/1024:>8.1f} KB')

print(f'\ntotal samples : {len(df):,}')
print(f'num classes   : {len(le.classes_)}')
print(f'classes       : {list(le.classes_)}')

processed files:
  label_encoder.pkl              0.3 KB
  test.csv                    6866.4 KB
  train.csv                  31455.0 KB
  val.csv                     6676.5 KB
  cleaned_data.csv           44997.8 KB

total samples : 47,371
num classes   : 5
classes       : ['Anxiety/Stress', 'Bipolar', 'Depression', 'Normal', 'Suicidal']
